# TRAITEMENT DE DONNÉES

### PROBLEMATIQUE DU SUJET : La modélisation des causes et les caractéristiques des accidents pour réduire leur nombre et leur gravité

## Importation des librairies

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split

## Importation des bases de données

In [2]:
def nettoyer_colonnes(df):

    """

    Nettoie les noms de colonnes d'un DataFrame.
 
    Transformations appliquées :

    - mise en minuscules

    - remplacement des points (.) par des underscores (_)

    """
 
    # Nettoyage des noms de colonnes

    df.columns = (

        df.columns

            .str.lower()                     # tout en minuscules

            .str.replace('.', '_')           # remplace les points par _
    )
 
    return df

* **Base caracteristique**

In [3]:
df_caract = pd.read_csv("../data_bases/caracteristique.txt", sep="\t")  # sep="," si CSV classique
df_caract.head(10)

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,3,JEAN MOULIN (RUE) - RIV. DES GALETS,"-20,95888","55,31743"
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,2,SAINT ROBERT (RUE),"45,79137","3,1162"
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,1,POISSOMPRE (FG DE),"48,18261","6,47218"
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,3,CAGNARD (RUE),"49,90355","2,2872"
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,1,A7-Sens Lyon vers Marseille,"43,34031","5,37689"
5,202100048274,12,3,2021,16:25,1,CC_30189,2,2,1,6,TALABOT (BOULEVARD),"43,83843726","4,372475088"
6,202100054416,18,1,2021,15:25,1,CC_91228,2,3,1,2,BD CHAMPS ELYSEES,"48,633158","2,425203"
7,202100026295,29,7,2021,17:35,1,CC_51230,2,1,1,7,Rue Pierre Semard,"49,04566","3,96145"
8,202100010947,26,10,2021,09:10,1,CC_93059,2,3,1,3,MERCHER (PASSAGE),"48,94933","2,35775"
9,202100009689,4,11,2021,08:35,1,CC_55369,1,1,5,6,D28,"48,808277","5,211725"


In [4]:
df_caract.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56530 entries, 0 to 56529
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Num_Acc  56530 non-null  int64 
 1   jour     56530 non-null  int64 
 2   mois     56530 non-null  int64 
 3   an       56530 non-null  int64 
 4   hrmn     56530 non-null  object
 5   lum      56530 non-null  int64 
 6   com      56530 non-null  object
 7   agg      56530 non-null  int64 
 8   int      56530 non-null  int64 
 9   atm      56530 non-null  int64 
 10  col      56530 non-null  int64 
 11  adr      55957 non-null  object
 12  lat      56530 non-null  object
 13  long     56530 non-null  object
dtypes: int64(9), object(5)
memory usage: 6.0+ MB


In [5]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean puis remet correctement ce qu'il lui correspond
df_caract = df_caract.convert_dtypes()

In [6]:
def explore_base(df_caract):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_caract.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_caract.isna().sum().sort_values(ascending=False))

    for i in df_caract.columns:
        if df_caract[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_caract[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_caract[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_caract[i].value_counts(dropna=False))

In [7]:
explore_base(df_caract)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56530 entries, 0 to 56529
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Num_Acc  56530 non-null  Int64 
 1   jour     56530 non-null  Int64 
 2   mois     56530 non-null  Int64 
 3   an       56530 non-null  Int64 
 4   hrmn     56530 non-null  string
 5   lum      56530 non-null  Int64 
 6   com      56530 non-null  string
 7   agg      56530 non-null  Int64 
 8   int      56530 non-null  Int64 
 9   atm      56530 non-null  Int64 
 10  col      56530 non-null  Int64 
 11  adr      55957 non-null  string
 12  lat      56530 non-null  string
 13  long     56530 non-null  string
dtypes: Int64(9), string(5)
memory usage: 6.5 MB
None
----------------------------------
Contrôle des valeurs manquantes
----------------------------------
adr        573
Num_Acc      0
jour         0
mois         0
an           0
hrmn         0
lum        

* **Base Lieux**

In [8]:
df_lieux = pd.read_excel("../data_bases/lieux.xls")
df_lieux.head(10)

,Num_Acc,dep,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202100022009,40,1,A63,0,A,3,3,0,1,141,600,1,NaN,-1.0,1,0,1,130
1,202100023746,94,4,VLADIMIR ILLITCH LENINE (AV),0,NaN,-1,2,0,1,0,0,1,NaN,-1.0,1,0,1,50
2,202100019549,2A,2,20,0,T,2,2,0,1,-1,-1,2,NaN,-1.0,1,0,3,80
3,202100022748,75,4,AVENUE SIMON BOLIVAR,0,NaN,2,2,0,2,-1,-1,1,NaN,-1.0,1,0,1,50
4,202100051993,971,3,BOLOGNE (ROUTE DE),0,NaN,2,2,0,1,-1,-1,1,NaN,-1.0,1,0,1,50
5,202100038809,57,3,63,0,A,2,2,0,1,-1,170,1,NaN,-1.0,2,0,1,50
6,202100023370,971,4,NaN,0,NaN,2,1,1,1,-1,-1,1,NaN,3.0,1,0,3,50
7,202100038829,17,3,939,-1,NaN,2,2,0,1,28,659,1,NaN,-1.0,2,0,1,80
8,202100013461,66,3,916,0,NaN,2,2,0,1,6,645,1,NaN,-1.0,1,0,1,70
9,202100017993,78,3,GENERAL DE GAULLE (RUE DU),0,NaN,-1,0,0,1,38,662,1,NaN,-1.0,1,9,1,50


In [9]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean
df_lieux = df_lieux.convert_dtypes()

In [10]:
def explore_base(df_lieux):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_lieux.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_lieux.isna().sum().sort_values(ascending=False))

    for i in df_lieux.columns:
        if df_lieux[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_lieux[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_lieux[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_lieux[i].value_counts(dropna=False))

In [11]:
explore_base(df_lieux)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56525 entries, 0 to 56524
Data columns (total 19 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Num_Acc  56525 non-null  Int64  
 1   dep      56525 non-null  object 
 2   catr     56525 non-null  Int64  
 3   voie     52117 non-null  object 
 4   v1       56525 non-null  Int64  
 5   v2       4990 non-null   object 
 6   circ     56525 non-null  Int64  
 7   nbv      56525 non-null  Int64  
 8   vosp     56525 non-null  Int64  
 9   prof     56525 non-null  Int64  
 10  pr       56525 non-null  Int64  
 11  pr1      56525 non-null  Int64  
 12  plan     56525 non-null  Int64  
 13  lartpc   108 non-null    Float64
 14  larrout  56525 non-null  Float64
 15  surf     56525 non-null  Int64  
 16  infra    56525 non-null  Int64  
 17  situ     56525 non-null  Int64  
 18  vma      56525 non-null  Int64  
dtypes: Float64(2), Int64(14), object(3)
memory usa

* **Base usagers**

In [12]:
df_usag = pd.read_csv("../data_bases/usagers.csv")
df_usag.head(10)

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100021165,163Â 830,A01,9,2,4,1,1948.0,0,1,-1,-1,0,0,-1
1,202100020281,165Â 383,A01,1,1,1,2,1965.0,1,1,8,-1,-1,-1,-1
2,202100047336,117Â 153,A01,1,1,4,2,1972.0,0,2,0,-1,-1,-1,-1
3,202100008918,185Â 779,B01,1,1,4,1,1985.0,9,8,0,-1,-1,-1,-1
4,202100009175,185Â 315,A01,1,1,3,1,1960.0,5,1,-1,-1,0,0,-1
5,202100050277,111Â 849,B01,1,1,4,1,1972.0,3,1,9,-1,0,0,-1
6,202100027012,153Â 433,A01,1,1,1,1,1986.0,5,1,-1,-1,0,0,-1
7,202100028064,151Â 549,A01,1,1,1,2,1957.0,5,1,-1,-1,0,0,-1
8,202100003701,195Â 137,A01,1,1,4,2,2006.0,0,2,0,-1,-1,-1,-1
9,202100033813,141Â 320,A01,1,1,1,1,1999.0,5,1,8,-1,-1,-1,-1


In [13]:
# 1) Corriger l'affichage du caractère A dans id_vehicule
df_usag["id_vehicule"] = df_usag["id_vehicule"].str.replace("Â", "A", regex=False)

# 2) Supprimer le .0 dans an_nais (ex: 1948.0 → 1948)
df_usag["an_nais"] = df_usag["an_nais"].astype("Int64")
df_usag.head(10)

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100021165,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
1,202100020281,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1
2,202100047336,117A 153,A01,1,1,4,2,1972,0,2,0,-1,-1,-1,-1
3,202100008918,185A 779,B01,1,1,4,1,1985,9,8,0,-1,-1,-1,-1
4,202100009175,185A 315,A01,1,1,3,1,1960,5,1,-1,-1,0,0,-1
5,202100050277,111A 849,B01,1,1,4,1,1972,3,1,9,-1,0,0,-1
6,202100027012,153A 433,A01,1,1,1,1,1986,5,1,-1,-1,0,0,-1
7,202100028064,151A 549,A01,1,1,1,2,1957,5,1,-1,-1,0,0,-1
8,202100003701,195A 137,A01,1,1,4,2,2006,0,2,0,-1,-1,-1,-1
9,202100033813,141A 320,A01,1,1,1,1,1999,5,1,8,-1,-1,-1,-1


In [14]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean
df_usag = df_usag.convert_dtypes()

In [15]:
def explore_base(df_usag):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_usag.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_usag.isna().sum().sort_values(ascending=False))

    for i in df_usag.columns:
        if df_usag[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_usag[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_usag[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_usag[i].value_counts(dropna=False))

In [16]:
explore_base(df_usag)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129189 entries, 0 to 129188
Data columns (total 15 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Num_Acc      129189 non-null  Int64 
 1   id_vehicule  129189 non-null  string
 2   num_veh      129189 non-null  string
 3   place        129189 non-null  Int64 
 4   catu         129189 non-null  Int64 
 5   grav         129189 non-null  Int64 
 6   sexe         129189 non-null  Int64 
 7   an_nais      126122 non-null  Int64 
 8   trajet       129189 non-null  Int64 
 9   secu1        129189 non-null  Int64 
 10  secu2        129189 non-null  Int64 
 11  secu3        129189 non-null  Int64 
 12  locp         129189 non-null  Int64 
 13  actp         129189 non-null  string
 14  etatp        129189 non-null  Int64 
dtypes: Int64(12), string(3)
memory usage: 16.3 MB
None
----------------------------------
Contrôle des valeurs manquantes
----------

* **Base vehicules**

In [17]:
df_veh = pd.read_csv("../data_bases/vehicules.csv",sep=";", encoding="latin-1", decimal=",")
df_veh.head(10)

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,129åÊ983,A01,1,33,0,2,3,1,1,NaN
1,192åÊ850,B01,1,10,0,2,4,2,1,NaN
2,152åÊ776,A01,1,33,0,2,1,3,3,NaN
3,193åÊ398,A01,1,7,0,1,2,1,1,NaN
4,178åÊ605,A01,2,7,0,1,2,1,1,NaN
5,187åÊ823,A01,2,30,0,2,1,1,1,NaN
6,103åÊ719,B01,1,1,0,2,1,1,5,NaN
7,164åÊ572,B01,1,7,8,2,2,1,1,NaN
8,190åÊ438,B01,3,15,0,2,3,1,1,NaN
9,146åÊ391,B01,1,7,0,2,4,2,1,NaN


In [18]:
# Modification format id_vehicule
df_veh["id_vehicule"]=df_veh["id_vehicule"].str.replace("åÊ", "A", regex=False)
df_veh.head()

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,129A983,A01,1,33,0,2,3,1,1,NaN
1,192A850,B01,1,10,0,2,4,2,1,NaN
2,152A776,A01,1,33,0,2,1,3,3,NaN
3,193A398,A01,1,7,0,1,2,1,1,NaN
4,178A605,A01,2,7,0,1,2,1,1,NaN


In [19]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean
df_veh = df_veh.convert_dtypes()

In [20]:
def explore_base(df_veh):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_veh.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_veh.isna().sum().sort_values(ascending=False))

    for i in df_veh.columns:
        if df_veh[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_veh[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_veh[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_veh[i].value_counts(dropna=False))

In [21]:
explore_base(df_veh)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97365 entries, 0 to 97364
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id_vehicule  97365 non-null  string
 1   num_veh      97365 non-null  string
 2   senc         97365 non-null  Int64 
 3   catv         97365 non-null  Int64 
 4   obs          97365 non-null  Int64 
 5   obsm         97365 non-null  Int64 
 6   choc         97365 non-null  Int64 
 7   manv         97365 non-null  Int64 
 8   motor        97365 non-null  Int64 
 9   occutc       744 non-null    Int64 
dtypes: Int64(8), string(2)
memory usage: 8.2 MB
None
----------------------------------
Contrôle des valeurs manquantes
----------------------------------
occutc         96621
id_vehicule        0
num_veh            0
senc               0
catv               0
obs                0
obsm               0
choc               0
manv               0
motor       

## Exporter les bases propres 

In [81]:
# On remonte d'un seul niveau pour atteindre la racine
df_caract_clean.to_csv('../data_bases/caract_clean.csv', sep=';', index=False, encoding='utf-8-sig')
df_lieux.to_csv('../data_bases/lieux_clean.csv', sep=';', index=False, encoding='utf-8-sig')
df_usag_clean.to_csv('../data_bases/usag_clean.csv', sep=';', index=False, encoding='utf-8-sig')
df_veh_clean.to_csv('../data_bases/veh_clean.csv', sep=';', index=False, encoding='utf-8-sig')

# Base finale
df_final.to_csv('../data_bases/base_complete.csv', sep=';', index=False, encoding='utf-8-sig')

# Vérification des doublons et nettoyage des données dans chaque base

* **Base caracteristique**

**Traitement des doublons pures et impures**

In [22]:
df_caract.duplicated().sum()

np.int64(11)

*Doublons pure*

In [23]:
df_caract[df_caract.duplicated(keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
33687,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
6317,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
25206,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
19025,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
32639,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
420,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
45121,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
50662,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
2994,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"
55470,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"


*Doublons inpures*

In [24]:
df_caract[df_caract.duplicated(subset=["Num_Acc"], keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
33687,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
6317,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
25206,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
19025,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
32639,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
420,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
45121,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
50662,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
2994,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"
55470,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"


**Commentaire :**  Dans la base caracteristique, on constate la présence de **11** doublons qui sont pures et inpures en même temps

In [25]:
# Supprime les doublons en gardant la première occurrence
df_caract = df_caract.drop_duplicates(subset=["Num_Acc"], keep='first')

*Verfication à nouveau*

In [26]:
df_caract[df_caract.duplicated(subset=["Num_Acc"], keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long


In [27]:
df_caract[df_caract.duplicated(keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long


In [28]:
df_caract.duplicated().sum()

np.int64(0)

In [29]:
# Verfication de numéro unique
print(df_caract.Num_Acc.nunique())

56519


In [30]:
# Affichage de la base après traitement
df_caract_clean=df_caract
df_caract_clean

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,3,JEAN MOULIN (RUE) - RIV. DES GALETS,"-20,95888","55,31743"
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,2,SAINT ROBERT (RUE),"45,79137","3,1162"
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,1,POISSOMPRE (FG DE),"48,18261","6,47218"
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,3,CAGNARD (RUE),"49,90355","2,2872"
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,1,A7-Sens Lyon vers Marseille,"43,34031","5,37689"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56525,202100026753,28,7,2021,17:10,1,CC_38474,2,2,1,3,Rue du Vinay,"45,204957","5,667561"
56526,202100019814,10,9,2021,06:15,3,CC_77251,1,1,1,7,RN 104,"48,631","2,533"
56527,202100016048,29,9,2021,08:15,1,CC_78524,1,1,1,3,A13,"48,841059","2,106997"
56528,202100039830,15,5,2021,06:45,2,CC_13212,2,2,1,3,PIERRE CHEVALIER (avenue),"43,2953","5,4254"


* **Base lieux**

In [31]:
# Verification des doublons
df_lieux.duplicated().sum()

np.int64(0)

**Commentaire :**  Dans la base lieux, on constate la présence de **0** doublons

**Remplacer NaN de la colonne v2 par Inconnu**

Pour les colonnes textuelles comme voie tel que **v2** qui est la lettre de l'indice alphanumérique de la route est importante, donc j'ai remplacé par "Inconnu"

In [32]:
df_lieux["v2"] = df_lieux["v2"].fillna("Inconnu")
df_lieux["v2"]

0              A
1        Inconnu
2              T
3        Inconnu
4        Inconnu
          ...   
56520    Inconnu
56521    Inconnu
56522    Inconnu
56523    Inconnu
56524    Inconnu
Name: v2, Length: 56525, dtype: object

In [33]:
print(df_lieux["v2"].value_counts())

v2
Inconnu    51535
D           3164
A            896
N            225
C            119
E            106
R             92
B             85
M             74
 -            67
T             43
F             24
X             15
V             14
O             12
U              9
G              8
S              7
Z              6
L              5
P              5
H              3
K              3
J              3
W              2
I              2
34             1
Name: count, dtype: int64


**Gérer les NaN de la colonnes lartpc**

**lartpc** est la largeur du terre-plein central (TPC) (en m), elle contient plus de **99%** de **NaN**, or elle n'apporte aucune valeur statistique. Donc je l'a supprime

In [34]:
# Calcul du pourcentage de NaN pour lartpc
pct_nan_lartpc = df_lieux['lartpc'].isnull().mean() * 100

print(f"Le pourcentage de valeurs manquantes dans 'lartpc' est de : {pct_nan_lartpc:.2f}%")

Le pourcentage de valeurs manquantes dans 'lartpc' est de : 99.81%


In [35]:
# Supprime les colonnes qui ont trop de vides
limit = len(df_lieux) * 0.5  # Seuil de 50%
df_lieux = df_lieux.dropna(thresh=limit, axis=1)

In [36]:
# Affichage de la base propre
df_lieux_clean=df_lieux
df_lieux_clean

,Num_Acc,dep,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,larrout,surf,infra,situ,vma
0,202100022009,40,1,A63,0,A,3,3,0,1,141,600,1,-1.0,1,0,1,130
1,202100023746,94,4,VLADIMIR ILLITCH LENINE (AV),0,Inconnu,-1,2,0,1,0,0,1,-1.0,1,0,1,50
2,202100019549,2A,2,20,0,T,2,2,0,1,-1,-1,2,-1.0,1,0,3,80
3,202100022748,75,4,AVENUE SIMON BOLIVAR,0,Inconnu,2,2,0,2,-1,-1,1,-1.0,1,0,1,50
4,202100051993,971,3,BOLOGNE (ROUTE DE),0,Inconnu,2,2,0,1,-1,-1,1,-1.0,1,0,1,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56520,202100056035,76,4,LYONS. ROUTE DE (BRETELLE C),0,Inconnu,1,2,0,1,-1,-1,1,-1.0,1,-1,1,30
56521,202100043915,75,4,RUE LOUIS BLANC,0,Inconnu,1,2,0,1,-1,-1,1,-1.0,1,0,1,30
56522,202100047332,73,4,JEUX OLYMPIQUES (AVENUE DES),0,Inconnu,3,2,2,1,-1,-1,1,-1.0,1,-1,1,50
56523,202100016135,64,4,SERGENT BERNES CAMBOT DU,0,Inconnu,2,3,1,1,0,0,1,-1.0,1,0,1,30


* **Base usagers**

**Traitement des doublons pures et impures**

In [37]:
# Affichage des doublons
df_usag.duplicated().sum()

np.int64(33)

In [38]:
df_usag.duplicated(subset=["id_vehicule"]).sum()

np.int64(31880)

In [39]:
df_usag["id_vehicule"].value_counts().loc[lambda x: x > 1]

id_vehicule
196A 125    21
172A 295    17
169A 915    16
155A 083    14
131A 903    14
            ..
168A 706     2
152A 274     2
145A 970     2
143A 669     2
196A 684     2
Name: count, Length: 23989, dtype: Int64

**Doublons pures**

In [40]:
df_usag[df_usag.duplicated(keep=False)].sort_values("id_vehicule")

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
24115,202100052486,107A 904,A01,1,1,3,1,1990,5,1,-1,-1,0,0,-1
9691,202100052486,107A 904,A01,1,1,3,1,1990,5,1,-1,-1,0,0,-1
48087,202100052378,108A 090,B01,1,1,4,1,2001,0,2,0,-1,-1,-1,-1
118307,202100052378,108A 090,B01,1,1,4,1,2001,0,2,0,-1,-1,-1,-1
4157,202100050391,111A 622,B01,1,1,1,1,1975,4,1,-1,-1,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77041,202100006907,189A 424,A01,2,2,4,1,1998,5,1,0,-1,-1,-1,-1
28399,202100003260,195A 936,A01,10,3,4,2,2002,5,0,-1,-1,3,3,2
108756,202100003260,195A 936,A01,10,3,4,2,2002,5,0,-1,-1,3,3,2
72758,202100001471,199A 156,B01,1,1,1,1,1995,9,1,0,-1,-1,-1,-1


**Commentaire :** Dans la base usagers, je constate la présence de 33 doublons

In [41]:
# Supprime uniquement les doublons exacts sur toutes les colonnes 
df_usag = df_usag.drop_duplicates(keep='first')  # garde la première occurrence
df_usag

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100021165,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
1,202100020281,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1
2,202100047336,117A 153,A01,1,1,4,2,1972,0,2,0,-1,-1,-1,-1
3,202100008918,185A 779,B01,1,1,4,1,1985,9,8,0,-1,-1,-1,-1
4,202100009175,185A 315,A01,1,1,3,1,1960,5,1,-1,-1,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129184,202100018230,169A 018,D01,1,1,1,2,1982,5,1,-1,-1,0,0,-1
129185,202100050268,111A 866,B01,2,2,4,2,1942,1,1,-1,-1,0,0,-1
129186,202100056519,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
129187,202100056520,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1


In [42]:
print(df_usag.shape)

(129156, 15)


**Doublons inpures**

In [43]:
df_usag[df_usag.duplicated(subset=["Num_Acc"], keep=False)].sort_values("Num_Acc")

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
24992,202100000001,201A 765,A01,1,1,1,1,1978,1,1,-1,-1,0,0,-1
32123,202100000001,201A 764,B01,1,1,3,1,2000,1,0,9,-1,0,0,-1
14610,202100000002,201A 763,B01,1,1,3,1,1993,0,1,-1,-1,0,0,-1
45309,202100000002,201A 762,A01,1,1,4,1,1983,0,1,-1,-1,0,0,-1
88804,202100000003,201A 761,A01,10,3,3,2,1959,4,0,-1,-1,3,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1891,202100056513,100A 890,B01,1,1,4,2,1971,5,1,-1,-1,0,0,-1
79646,202100056516,100A 886,B01,1,1,4,1,2002,5,1,-1,-1,0,0,-1
88203,202100056516,100A 885,A01,1,1,4,1,1975,0,1,-1,-1,0,0,-1
26063,202100056518,100A 882,A01,1,1,3,1,1968,3,1,0,-1,-1,-1,-1


In [44]:
# Pour voir s'il y a des vraies erreurs (même accident, même véhicule, même place)
df_usag[df_usag.duplicated(subset=["Num_Acc", "num_veh", "id_vehicule", "place", "an_nais", "sexe"], keep=False)]

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
1437,202100032490,143A 693,A01,10,3,4,2,1990,2,0,-1,-1,3,3,1
2414,202100000247,201A 281,A01,10,3,4,1,1980,5,0,-1,-1,5,0,-1
2785,202100053070,106A 861,A01,1,2,4,1,1999,5,-1,0,-1,-1,-1,-1
6223,202100029471,149A 107,A01,10,3,3,2,2004,5,0,-1,-1,5,1,3
6842,202100004108,194A 394,A01,2,2,1,1,2006,9,8,0,-1,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126119,202100030699,146A 886,A01,10,3,3,2,2009,5,0,-1,-1,3,3,3
126217,202100050659,111A 133,A01,5,2,1,1,2005,0,8,0,-1,-1,-1,-1
126591,202100013335,177A 911,A01,10,3,1,1,1973,0,0,-1,-1,5,1,2
128052,202100024185,158A 436,A01,10,3,4,1,1998,5,0,-1,-1,6,1,3


In [45]:
df_usag.isna().sum()

Num_Acc           0
id_vehicule       0
num_veh           0
place             0
catu              0
grav              0
sexe              0
an_nais        3067
trajet            0
secu1             0
secu2             0
secu3             0
locp              0
actp              0
etatp             0
dtype: int64

In [46]:
df_usag[df_usag['an_nais'].isna()].sort_values('id_vehicule')

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
127037,202100056500,100A 909,A01,1,1,1,-1,<NA>,-1,-1,-1,-1,-1,-1,-1
18360,202100056487,100A 930,Z01,1,1,1,-1,<NA>,-1,0,0,0,-1,-1,-1
25321,202100056485,100A 935,Z01,1,1,1,-1,<NA>,-1,-1,-1,-1,-1,-1,-1
66525,202100056472,100A 959,B01,1,1,1,-1,<NA>,-1,-1,-1,-1,-1,-1,-1
8253,202100056439,101A 015,A01,1,1,1,-1,<NA>,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93622,202100000051,201A 657,B01,1,1,1,-1,<NA>,-1,-1,-1,-1,-1,-1,-1
8488,202100000045,201A 676,A01,1,1,1,-1,<NA>,-1,-1,-1,-1,-1,-1,-1
48815,202100000038,201A 694,A01,1,1,1,-1,<NA>,-1,8,-1,-1,0,0,-1
52373,202100000020,201A 725,A01,1,1,1,-1,<NA>,-1,-1,-1,-1,-1,-1,-1


**3067** ne sont pas des doublons inpures, ce sont les années de naissance

In [47]:
df_usag.dropna(subset=['an_nais'], inplace=True)

In [48]:
df_usag[df_usag['sexe'].isin([0, -1])]

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
49978,202100021078,163A 980,B01,1,1,1,-1,1993,-1,8,8,8,0,0,-1
56737,202100034601,139A 865,A01,1,1,1,-1,1992,-1,8,8,8,0,0,-1


In [49]:
df_usag = df_usag[~df_usag['sexe'].isin([0, -1])]

In [50]:
print(df_usag.id_vehicule.nunique())
print(df_usag.shape)

95188
(126087, 15)


In [51]:
# Affichage de la base propre
df_usag_clean=df_usag
df_usag_clean

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100021165,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
1,202100020281,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1
2,202100047336,117A 153,A01,1,1,4,2,1972,0,2,0,-1,-1,-1,-1
3,202100008918,185A 779,B01,1,1,4,1,1985,9,8,0,-1,-1,-1,-1
4,202100009175,185A 315,A01,1,1,3,1,1960,5,1,-1,-1,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129184,202100018230,169A 018,D01,1,1,1,2,1982,5,1,-1,-1,0,0,-1
129185,202100050268,111A 866,B01,2,2,4,2,1942,1,1,-1,-1,0,0,-1
129186,202100056519,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
129187,202100056520,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1


* **Base vehicules**

In [52]:
df_veh.duplicated().sum()

np.int64(0)

**Commentaire :**  Dans la base vehicule, je constate la présence de **0** doublons

In [53]:
# Vérification s'il y a au moins une valeur remplie
print(df_veh['occutc'].notnull().sum())

744


In [54]:
# Remplacer les NaN par 0
df_veh['occutc'] = df_veh['occutc'].fillna(0)

# Convertir en entier (pour enlever le .0)
df_veh['occutc'] = df_veh['occutc'].astype(int)

In [55]:
# Affichage de la base propre
df_veh_clean=df_veh
df_veh_clean

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,129A983,A01,1,33,0,2,3,1,1,0
1,192A850,B01,1,10,0,2,4,2,1,0
2,152A776,A01,1,33,0,2,1,3,3,0
3,193A398,A01,1,7,0,1,2,1,1,0
4,178A605,A01,2,7,0,1,2,1,1,0
...,...,...,...,...,...,...,...,...,...,...
97360,108A341,A01,1,1,0,0,0,1,5,0
97361,200A693,B01,1,1,0,2,5,26,5,0
97362,192A529,B01,1,31,0,2,1,1,1,0
97363,174A733,B01,1,2,0,2,3,1,1,0


In [56]:
# Affichage des valeurs differentes de 0 dans la colonne occutc
df_veh[df_veh['occutc']!=0]

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
74,172A370,A01,1,37,0,2,3,0,1,1
125,145A258,B01,1,37,0,2,6,1,1,1
224,154A771,B01,1,37,0,2,7,1,1,1
262,178A037,B01,2,37,0,2,4,1,1,1
278,110A314,B01,2,38,0,2,3,1,1,1
...,...,...,...,...,...,...,...,...,...,...
97025,130A817,B01,2,37,0,2,8,0,1,1
97112,151A326,A01,1,37,0,2,3,19,1,4
97150,155A428,B01,1,37,0,0,8,15,1,1
97184,179A775,B01,2,37,0,2,4,23,6,1


# Jointure des bases 

In [57]:
# Verification du nombre de lignes et colonne restantes dans chaque base
print(df_caract_clean.shape)
print(df_lieux_clean.shape)
print(df_usag_clean.shape)
print(df_veh_clean.shape)

(56519, 14)
(56525, 18)
(126087, 15)
(97365, 10)


In [58]:
# Jointure des tables df_caract et  df_lieux
df_caract_lieux= df_caract_clean.merge(df_lieux_clean,left_on="Num_Acc",right_on="Num_Acc",how="left")
df_caract_lieux.head()

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,vosp,prof,pr,pr1,plan,larrout,surf,infra,situ,vma
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,0,1,0,0,1,-1.0,1,5,1,50
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,0,1,0,0,1,-1.0,1,0,1,50
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,...,0,1,0,920,1,-1.0,1,0,1,50
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,...,0,4,-1,-1,1,-1.0,1,0,1,50
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,...,0,2,279,0,2,-1.0,1,0,1,70


In [59]:
df_caract_lieux.shape

(56526, 31)

In [60]:
df_caract_lieux[(df_caract_lieux["Num_Acc"].isna())]

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,vosp,prof,pr,pr1,plan,larrout,surf,infra,situ,vma


In [61]:
# Jointure des tables df_caract_lieux et df_usag
df_caract_lieux_usag= df_caract_lieux.merge(df_usag_clean,left_on="Num_Acc",right_on="Num_Acc",how="inner")
df_caract_lieux_usag.head()

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,4,1,1970,0,8,8,-1,-1,-1,-1
1,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,1,1,1998,0,1,0,-1,-1,-1,-1
2,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,1,1,1979,1,1,0,-1,0,0,-1
3,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,4,1,1995,0,1,0,-1,0,0,-1
4,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,...,1,1,1931,5,1,0,-1,-1,-1,-1


In [62]:
df_caract_lieux_usag.shape

(126097, 45)

In [63]:
df_caract_lieux_usag[(df_caract_lieux_usag["Num_Acc"].isna())]

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp


In [64]:
# Affichage des données dans la colonne id_vehicule
df_caract_lieux_usag['id_vehicule']

0         134A 220
1         134A 221
2         113A 491
3         113A 490
4         132A 115
            ...   
126092    172A 965
126093    130A 384
126094    130A 383
126095    118A 220
126096    118A 220
Name: id_vehicule, Length: 126097, dtype: string

In [67]:
df_veh['id_vehicule']

0        129A983
1        192A850
2        152A776
3        193A398
4        178A605
          ...   
97360    108A341
97361    200A693
97362    192A529
97363    174A733
97364    181A572
Name: id_vehicule, Length: 97365, dtype: string

In [68]:
# Nettoyage des espaces internes et mise en majuscules pour les deux tables
df_caract_lieux_usag['id_vehicule'] = df_caract_lieux_usag['id_vehicule'].astype(str).str.replace(r'\s+', '', regex=True).str.upper()
df_veh['id_vehicule'] = df_veh['id_vehicule'].astype(str).str.replace(r'\s+', '', regex=True).str.upper()

Le symbole **\s+** en expression régulière (**regex=True**) détecte tous les caractères "blancs" (espaces, tabulations, retours à la ligne) et les supprime.

In [70]:
# Verification après correction
df_caract_lieux_usag['id_vehicule']

0         134A220
1         134A221
2         113A491
3         113A490
4         132A115
           ...   
126092    172A965
126093    130A384
126094    130A383
126095    118A220
126096    118A220
Name: id_vehicule, Length: 126097, dtype: object

In [69]:
# la jointure des 4 bases de données
df_final = df_caract_lieux_usag.merge(df_veh, on='id_vehicule', how='inner')

print(f"Nombre de lignes dans la base finale : {len(df_final)}")

Nombre de lignes dans la base finale : 126097


In [71]:
#Affichage de la jointure de la base finale
df_final

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,etatp,num_veh_y,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,-1,B01,3,1,0,2,8,1,5,0
1,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,-1,A01,3,7,0,9,1,1,1,0
2,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,-1,A01,1,10,0,2,1,2,1,0
3,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,-1,B01,1,7,0,2,4,2,1,0
4,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,...,-1,A01,1,7,0,2,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126092,202100016048,29,9,2021,08:15,1,CC_78524,1,1,1,...,-1,B01,2,7,0,2,2,12,1,0
126093,202100039830,15,5,2021,06:45,2,CC_13212,2,2,1,...,-1,A01,3,10,0,2,1,1,0,0
126094,202100039830,15,5,2021,06:45,2,CC_13212,2,2,1,...,-1,B01,3,7,0,2,5,1,1,0
126095,202100046742,13,3,2021,09:30,1,CC_76108,2,1,2,...,-1,A01,1,7,0,1,1,16,1,0


In [72]:
df_final.shape

(126097, 54)

In [73]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126097 entries, 0 to 126096
Data columns (total 54 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Num_Acc      126097 non-null  Int64  
 1   jour         126097 non-null  Int64  
 2   mois         126097 non-null  Int64  
 3   an           126097 non-null  Int64  
 4   hrmn         126097 non-null  string 
 5   lum          126097 non-null  Int64  
 6   com          126097 non-null  string 
 7   agg          126097 non-null  Int64  
 8   int          126097 non-null  Int64  
 9   atm          126097 non-null  Int64  
 10  col          126097 non-null  Int64  
 11  adr          124742 non-null  string 
 12  lat          126097 non-null  string 
 13  long         126097 non-null  string 
 14  dep          126097 non-null  object 
 15  catr         126097 non-null  Int64  
 16  voie         117057 non-null  object 
 17  v1           126097 non-null  Int64  
 18  v2           126097 non-

## Constitution des bases de train et de test

**la préparation de ta variable cible**

Dans le jeu de données BAAC, la colonne grav (gravité) possède 4 modalités :

1 : Indemne

2 : Tué

3 : Blessé hospitalisé

4 : Blessé léger

Pour le modèle de **Machine Learning**, il est souvent préférable de simplifier cela en une classification binaire (Grave vs Non-Grave).

In [74]:
# Vérification de la distribution de la gravité
print("Répartition des classes de gravité :")
print(df_final['grav'].value_counts())

# Vérification du nombre total de lignes et colonnes
print(f"\nDimensions finales : {df_final.shape}")

Répartition des classes de gravité :
grav
1    52105
4    51685
3    19088
2     3219
Name: count, dtype: Int64

Dimensions finales : (126097, 54)


**Création de la variable binaire**

Selon la nomenclature officielle, nous avons isoler les conséquences les plus lourdes :

- Grave (1) : Tués (2) et Blessés hospitalisés (3),

- Non-Grave (0) : Indemnes (1) et Blessés légers (4).

In [75]:
# Création de la fonction de mapping
def mapping_gravite(val):
    if val in [2, 3]: # Tué ou Blessé hospitalisé
        return 1
    else: # Indemne ou Blessé léger
        return 0

# Application sur une nouvelle colonne 'target'
df_final['target'] = df_final['grav'].apply(mapping_gravite)

# Vérification du nouvel équilibre
print("Répartition de la variable cible (Target) :")
print(df_final['target'].value_counts(normalize=True) * 100)

Répartition de la variable cible (Target) :
target
0    82.309651
1    17.690349
Name: proportion, dtype: float64


In [78]:
# On ne garde que les colonnes numériques ou catégorielles encodées, on supprime les identifiants et les colonnes textuelles (comme 'adr')
cols_exclues = ['Num_Acc', 'id_vehicule', 'num_veh', 'adr', 'grav']
df_prep = df_final.drop(columns=[c for c in cols_exclues if c in df_final.columns])

# Gestion des valeurs manquantes (remplacement par le mode ou 0)
df_prep = df_prep.fillna(0) 

# Conversion des colonnes "object" en catégories (si nécessaire)
for col in df_prep.select_dtypes(include=['object']).columns:
    df_prep[col] = df_prep[col].astype('category').cat.codes

In [79]:
# X = les caractéristiques (météo, route, véhicule)
# y = la cible (target : 0 ou 1)
X = df_prep.drop(columns=['target'])
y = df_prep['target']

# Division 80% Train / 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Base d'entraînement : {X_train.shape[0]} accidents")
print(f"Base de test : {X_test.shape[0]} accidents")

Base d'entraînement : 100877 accidents
Base de test : 25220 accidents


In [80]:
# On remonte de deux niveaux pour atteindre la racine, puis on entre dans data_bases
X_train.to_csv('./../data_bases/X_train.csv', index=False)
X_test.to_csv('./../data_bases/X_test.csv', index=False)
y_train.to_csv('./../data_bases/y_train.csv', index=False)
y_test.to_csv('./../data_bases/y_test.csv', index=False)

**************Fin traitement*************